## Ollama Embeddings + FAISS + Chatbot

In [2]:
!pip install requests beautifulsoup4 tqdm

Defaulting to user installation because normal site-packages is not writeable


In [3]:

from pathlib import Path
import os

PROJECT_ROOT = Path(
    r"C:\Users\aravi\Documents\RAG Azure document Assistant"
)

SERVICE_NAME = "Azure_Communication_Services"
DOCS_FOLDER = PROJECT_ROOT / "docs" / SERVICE_NAME
FAISS_FOLDER = PROJECT_ROOT / "faiss_index"

print("Project root exists:", PROJECT_ROOT.exists())
print("Documentation folder:", DOCS_FOLDER)
print("Documentation folder exists:", DOCS_FOLDER.exists())

if not DOCS_FOLDER.exists():
    raise FileNotFoundError(
        f"Documentation folder not found: {DOCS_FOLDER}"
    )

os.chdir(PROJECT_ROOT)
FAISS_FOLDER.mkdir(parents=True, exist_ok=True)

print("Markdown files:", len(list(DOCS_FOLDER.glob("*.md"))))


Project root exists: True
Documentation folder: C:\Users\aravi\Documents\RAG Azure document Assistant\docs\Azure_Communication_Services
Documentation folder exists: True
Markdown files: 32


In [4]:
import os
import pickle
import faiss
import numpy as np
import ollama

In [5]:
SERVICE_NAME="Azure_Communication_Services"

DOCS_FOLDER=os.path.join("docs",SERVICE_NAME)

FAISS_FOLDER="faiss_index"
os.makedirs(FAISS_FOLDER,exist_ok=True)

EMBED_MODEL="nomic-embed-text"
CHAT_MODEL="llama3.2"

In [6]:
documents=[]

for file in os.listdir(DOCS_FOLDER):
    if file.endswith(".md"):
        with open(os.path.join(DOCS_FOLDER,file),"r",encoding="utf-8") as f:
            documents.append({"file":file,"text":f.read()})

print(len(documents))

32


In [7]:
def split_text(text,chunk_size=1000,overlap=200):
    chunks=[]
    start=0

    while start<len(text):
        end=start+chunk_size
        chunks.append(text[start:end])
        start+=chunk_size-overlap

    return chunks

In [8]:
chunks=[]

for doc in documents:
    for chunk in split_text(doc["text"]):
        chunks.append({
            "file":doc["file"],
            "text":chunk
        })

print(len(chunks))

1146


In [9]:
embeddings=[]

for chunk in chunks:
    response=ollama.embed(
        model=EMBED_MODEL,
        input=chunk["text"]
    )

    embeddings.append(response["embeddings"][0])

embeddings=np.array(embeddings,dtype="float32")

In [10]:
dimension=embeddings.shape[1]

index=faiss.IndexFlatL2(dimension)

index.add(embeddings)

print(index.ntotal)

1146


In [11]:
faiss.write_index(
    index,
    os.path.join(
        FAISS_FOLDER,
        SERVICE_NAME+".index"
    )
)

with open(
    os.path.join(
        FAISS_FOLDER,
        SERVICE_NAME+"_chunks.pkl"
    ),
    "wb"
) as f:
    pickle.dump(chunks,f)

In [12]:
index=faiss.read_index(
    os.path.join(
        FAISS_FOLDER,
        SERVICE_NAME+".index"
    )
)

with open(
    os.path.join(
        FAISS_FOLDER,
        SERVICE_NAME+"_chunks.pkl"
    ),
    "rb"
) as f:
    chunks=pickle.load(f)

In [13]:
def retrieve_chunks(question,top_k=5):

    response=ollama.embed(
        model=EMBED_MODEL,
        input=question
    )

    query=np.array(
        [response["embeddings"][0]],
        dtype="float32"
    )

    distances,indices=index.search(query,top_k)

    return [chunks[i] for i in indices[0]]

In [14]:
def build_context(results):

    return "\n\n".join(
        chunk["text"] for chunk in results
    )

In [15]:
def generate_answer(question,context):

    prompt=f'''
You are an Azure Documentation Assistant.

Answer ONLY using the context.

If the answer is unavailable, say:
"I couldn't find this information in the documentation."

Context:
{context}

Question:
{question}
'''

    response=ollama.chat(
        model=CHAT_MODEL,
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    return response["message"]["content"]


In [16]:
def ask_question(question):

    results=retrieve_chunks(question)

    context=build_context(results)

    return generate_answer(question,context)

In [17]:
while True:

    question=input("Ask Question (exit to quit): ")

    if question.lower()=="exit":
        break

    print()

    print(ask_question(question))

    print("="*60)


To generate an access token in Azure Communication Services, you need to create a Communication Services identity and then use the "Issue a user access token" action in the Azure Communication Services Identity connector. 

You can specify the token scope as "VoIP" or "chat". The system generates an access token and its expiration time with the specified scope.

I couldn't find this information in the documentation.


In [18]:
import ollama

response = ollama.embed(
    model="nomic-embed-text",
    input="Azure Communication Services documentation"
)

print("Embedding generated successfully!")
print("Embedding dimensions:", len(response["embeddings"][0]))

Embedding generated successfully!
Embedding dimensions: 768
